# 06 · Lista de casa: boosting, Optuna e SHAP

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- O alvo é o da lista 05: prever se a avaliação recebeu **nota 5**.
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- O conjunto de teste não é usado nesta lista.
- Antes de entregar, use **Reiniciar sessão e executar tudo** (a execução completa leva alguns minutos).

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Taxa de aprendizado e número de árvores | 2,0 |
| 2 | Diagnóstico de sobreajuste | 2,0 |
| 3 | Busca com Optuna | 2,5 |
| 4 | Interpretação com SHAP | 2,5 |
| 5 | Quando não usar boosting | 1,0 |

In [ ]:
import importlib.util
import subprocess
import sys

faltando = [pacote for pacote in ("lightgbm", "optuna", "shap") if importlib.util.find_spec(pacote) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

import inspect
import warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False)
df["nota_5"] = (df["overall_rating"] == 5).astype(int)
df["submission_date"] = pd.to_datetime(df["submission_date"])
idade = 2018 - df["reviewer_birth_year"]
df["idade"] = idade.where(idade.between(14, 100))
df["hora"] = df["submission_date"].dt.hour
df["dia_semana"] = df["submission_date"].dt.dayofweek
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
df["palavras"] = df["texto"].str.split().str.len()
df["exclamacoes"] = df["texto"].str.count("!")
df["maiusculas"] = df["texto"].str.count(r"[A-ZÀ-Ý]") / df["texto"].str.len()

NUMERICAS = ["idade", "hora", "dia_semana", "palavras", "exclamacoes", "maiusculas"]
CATEGORICAS = ["site_category_lv1", "site_category_lv2", "product_brand", "reviewer_state", "reviewer_gender"]

# A nota e a recomendação ficam FORA de X: usá-las seria vazamento (aula 05).
X = df[NUMERICAS + CATEGORICAS + ["texto"]]
y = df["nota_5"]

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMENTE)
X_ajuste, X_valid, y_ajuste, y_valid = train_test_split(X_treino, y_treino, test_size=0.25, stratify=y_treino, random_state=SEMENTE)

pre = ColumnTransformer(
    [
        ("num", "passthrough", NUMERICAS),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, max_categories=60), CATEGORICAS),
        ("txt", CountVectorizer(max_features=300, binary=True), "texto"),
    ],
    sparse_threshold=0,
)
A = pre.fit_transform(X_ajuste).astype(np.float32)
V = pre.transform(X_valid).astype(np.float32)
nomes = ["palavra: " + n[5:] if n.startswith("txt__") else n[5:] for n in pre.get_feature_names_out()]


def auc(modelo, X_, y_):
    return roc_auc_score(y_, modelo.predict_proba(X_)[:, 1])


def perda(modelo, X_, y_):
    """Perda logarítmica: penaliza probabilidades erradas e confiantes."""
    return log_loss(y_, modelo.predict_proba(X_)[:, 1])


def avaliar_em(*pares):
    """Argumentos de avaliação do `fit` do LightGBM (o nome mudou em versões recentes)."""
    if "eval_X" in inspect.signature(lgb.LGBMClassifier.fit).parameters:
        return {"eval_X": tuple(X_ for X_, _ in pares), "eval_y": tuple(y_ for _, y_ in pares)}
    return {"eval_set": list(pares)}


print(f"ajuste {A.shape} · validação {V.shape} · nota 5 em {y_ajuste.mean():.1%} do ajuste")

## Questão 1 (2,0 pontos): taxa de aprendizado e número de árvores

Os dados de brinquedo abaixo têm uma variável e ruído de variância 0,25.

a) Complete `melhor_numero_de_arvores(taxa)`: treine um `GradientBoostingRegressor` com `n_estimators=1500`, `max_depth=2`, `learning_rate=taxa` e `random_state=0` nos dados de ajuste; com `staged_predict`, calcule o erro quadrático médio na **validação** depois de cada árvore; devolva o número de árvores de menor erro, esse erro, e o erro com as 1500 árvores.

b) Monte `tabela_q1`, um DataFrame com as colunas `taxa`, `arvores`, `erro_minimo` e `erro_1500`, para as taxas 1,0, 0,3, 0,1 e 0,03, nessa ordem.

In [ ]:
rng = np.random.default_rng(7)

def gerar(n):
    x = rng.uniform(-3, 3, n)
    return x.reshape(-1, 1), x * np.cos(2 * x) + rng.normal(0, 0.5, n)

Xb_ajuste, yb_ajuste = gerar(300)
Xb_valid, yb_valid = gerar(300)

In [ ]:
# ✏️ Questão 1
def melhor_numero_de_arvores(taxa):
    # SEU CÓDIGO AQUI
    ...
    return arvores, erro_minimo, erro_1500


# SEU CÓDIGO AQUI
tabela_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
assert list(tabela_q1.columns) == ["taxa", "arvores", "erro_minimo", "erro_1500"] and tabela_q1["taxa"].tolist() == [1.0, 0.3, 0.1, 0.03]
assert (tabela_q1["erro_minimo"] <= tabela_q1["erro_1500"]).all() and (tabela_q1["erro_minimo"] > 0.2).all()
assert tabela_q1["arvores"].between(1, 1500).all() and tabela_q1["arvores"].iloc[0] < tabela_q1["arvores"].iloc[-1]
print("Formato correto.")

**1c)** Descreva a relação entre a taxa e o número ideal de árvores na sua tabela. O erro mínimo muda muito de uma taxa para outra? Compare `erro_minimo` e `erro_1500` e explique por que nenhum erro fica abaixo de 0,25.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): diagnóstico de sobreajuste

a) Treine `solto`: um `lgb.LGBMClassifier` com `n_estimators=300`, `learning_rate=0.1`, `num_leaves=255`, `min_child_samples=1`, `random_state=SEMENTE` e `verbose=-1`, na matriz `A`, **sem** parada antecipada.

b) Treine `contido`: um LightGBM com parada antecipada (na validação, paciência de 50 árvores) e hiperparâmetros à sua escolha, de modo que a **perda logarítmica** de validação seja menor que a do modelo solto e a distância entre a AUC de ajuste e a de validação fique abaixo de 0,05.

c) Monte `tabela_q2` com uma linha por modelo (índice `"solto"` e `"contido"`) e as colunas `auc_ajuste`, `auc_valid`, `perda_ajuste` e `perda_valid`. Use as funções `auc` e `perda` da preparação.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
solto = ...
contido = ...
tabela_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert list(tabela_q2.index) == ["solto", "contido"] and list(tabela_q2.columns) == ["auc_ajuste", "auc_valid", "perda_ajuste", "perda_valid"]
assert solto.get_params()["num_leaves"] == 255 and solto.get_params()["min_child_samples"] == 1
assert getattr(contido, "best_iteration_", None), "o modelo contido deve usar parada antecipada"
assert tabela_q2.loc["contido", "perda_valid"] < tabela_q2.loc["solto", "perda_valid"], "a perda de validação do modelo contido deve ser menor"
assert tabela_q2.loc["contido", "auc_ajuste"] - tabela_q2.loc["contido", "auc_valid"] < 0.05, "a distância entre ajuste e validação ainda é grande"
print("Condições atendidas.")

**2d)** Que sinais de sobreajuste o modelo solto apresenta? Compare os dois modelos pela AUC de validação e pela perda de validação: as duas medidas contam a mesma história? O que isso diz sobre as probabilidades que o modelo solto produz?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,5 pontos): busca com Optuna

a) Calcule `auc_padrao`: a AUC de validação de um LightGBM com hiperparâmetros padrão, `learning_rate=0.1`, `n_estimators=3000` e parada antecipada com paciência de 50.

b) Escreva `objetivo_q3` e rode `estudo_q3` com **20 tentativas**, maximizando a AUC de validação. A busca deve cobrir **pelo menos quatro** hiperparâmetros, escolhidos e delimitados por você. Use `TPESampler(seed=SEMENTE)`.

c) Execute a célula que estima o ruído entre sementes.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
auc_padrao = ...

def objetivo_q3(tentativa):
    ...

estudo_q3 = ...

In [ ]:
# Ruído entre sementes: o mesmo modelo, três sorteios diferentes (não altere)
_notas = []
for _semente in range(3):
    _m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=_semente, verbose=-1).fit(A, y_ajuste)
    _notas.append(auc(_m, V, y_valid))
amplitude_ruido = max(_notas) - min(_notas)
print("AUC com três sementes:", np.round(_notas, 4), f"· amplitude {amplitude_ruido:.4f}")

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert len(estudo_q3.trials) >= 20, "rode pelo menos 20 tentativas"
assert len(estudo_q3.best_params) >= 4, "a busca deve cobrir pelo menos quatro hiperparâmetros"
assert estudo_q3.direction.name == "MAXIMIZE" and 0.85 < auc_padrao < 0.95
assert estudo_q3.best_value > auc_padrao - 0.003, "a busca ficou bem abaixo do modelo padrão: revise os intervalos"
print(f"Formato correto. Ganho da busca: {estudo_q3.best_value - auc_padrao:+.4f} · ruído entre sementes: {amplitude_ruido:.4f}")

**3d)** Justifique os intervalos que você escolheu para dois dos hiperparâmetros. O ganho da busca é maior que o ruído entre sementes? Considerando o tempo gasto, em que situação você rodaria essa busca em um projeto real, e em que situação não rodaria?

**3e)** Na lista 05 você treinou uma regressão logística com TF-IDF para este mesmo alvo. Compare a AUC de validação que obteve lá com a melhor desta lista. O que explica o resultado?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,5 pontos): interpretação com SHAP

Use o modelo `modelo_q4`, treinado na célula abaixo, e a amostra `V_amostra` (2000 avaliações da validação).

a) Calcule `explicacao_q4` com `shap.TreeExplainer` e atribua `nomes` a `explicacao_q4.feature_names`.

b) Monte `top10_q4`: uma `Series` com as 10 colunas de maior média do valor SHAP **absoluto**, em ordem decrescente, indexada pelo nome da coluna.

c) Desenhe o gráfico global (`shap.plots.beeswarm`) e o gráfico em cascata (`shap.plots.waterfall`) da avaliação da amostra com a **maior** probabilidade prevista de nota 5. Imprima o texto dessa avaliação (`textos_amostra`).

In [ ]:
modelo_q4 = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.1, random_state=SEMENTE, verbose=-1)
modelo_q4.fit(A, y_ajuste, **avaliar_em((V, y_valid)), callbacks=[lgb.early_stopping(50, verbose=False)])

_indices = np.random.default_rng(SEMENTE).choice(len(V), size=2000, replace=False)
V_amostra = V[_indices]
textos_amostra = X_valid["texto"].iloc[_indices].to_numpy()
prob_amostra = modelo_q4.predict_proba(V_amostra)[:, 1]
print(f"AUC de validação: {auc(modelo_q4, V, y_valid):.4f}")

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
explicacao_q4 = ...
top10_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert explicacao_q4.values.shape == V_amostra.shape, "calcule os valores SHAP para as 2000 avaliações da amostra"
assert list(explicacao_q4.feature_names) == nomes
assert isinstance(top10_q4, pd.Series) and len(top10_q4) == 10 and top10_q4.is_monotonic_decreasing and set(top10_q4.index) <= set(nomes)
_lo = np.log(prob_amostra / (1 - prob_amostra))
assert np.allclose(explicacao_q4.base_values + explicacao_q4.values.sum(axis=1), _lo, atol=1e-4), "os valores SHAP não reconstroem as previsões de modelo_q4"
assert top10_q4.iloc[0] >= np.abs(explicacao_q4.values).mean(axis=0).max() - 1e-9, "a primeira coluna do top 10 não é a de maior média"
print("Formato correto.")

**4d)** Com base no gráfico global: cite duas palavras cuja presença empurra a previsão para "nota 5" e duas que empurram para longe. Há alguma palavra **positiva** que afasta a previsão de nota 5? Como você explica?

**4e)** Explique, em linguagem que um gestor entenda, a previsão da avaliação escolhida no item (c): de onde o modelo parte e o que o leva até a probabilidade final.

**4f)** A equipe de produto propõe: "como *excelente* é uma das palavras que mais aumentam a chance de nota 5, vamos sugerir essa palavra no campo de texto para elevar as notas". Avalie a proposta.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): quando não usar boosting

a) Descreva duas situações concretas, de preferência da sua área de atuação, em que você **não** escolheria gradient boosting, e diga o que usaria no lugar.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._